# Finetuning different types of Transformer Models.

## 01. Causal Language Modeling (CLM)
Used by <b>decoder</b> models like GPT, this approach predicts the next token based on all previous tokens in the sequence. The model can only use context from the left (previous tokens) to predict the next token.

#### Load ELI5 Dataset
Start by loading the first 5000 examples from the <u>ELI5-Category</u> dataset with the 🤗 Datasets library. This’ll give you a chance to experiment and make sure everything works before spending more time training on the full dataset.

In [1]:
from datasets import load_dataset

eli5 = load_dataset("eli5_category", split="train[:5000]", trust_remote_code=True)

# Split the dataset
eli5 = eli5.train_test_split(test_size=0.2)

# Checking one example
eli5["train"][0]

/home/koh/hf-learn-nbs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'q_id': '5lyheh',
 'title': 'The saying "feed a cold, starve a fever". Just an old wise tale or is it accurate? If so what is happening physically?',
 'selftext': 'I ask because I currently have a cold and my girlfriend keeps reiterating this saying to me, but when I was younger my Grandma would also say this. I joined Reddit fairly recently and this is my favorite thread. Hoping some of you can help clarify!',
 'category': 'Biology',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dbzeahq'],
  'text': ["Its an old wife's tale. A fever is a natural response from your immune system. If you starve yourself during that you also starve your immune system. It just people have a tenancy to throw up and keep nothing down during a fever so they tend to feed them less/they eat less. Not throwing up is important because it dehydrates you."],
  'score': [4],
  'text_urls': [[]]},
 'title_urls': ['url'],
 'selftext_urls': ['url']}

#### Preprocess

In [2]:
# Load DistilGPT2
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilgpt2"
)

You’ll notice from the example above, the <u>text</u> field is actually nested inside <u>answers</u>. This means you’ll need to extract the <u>text</u> subfield from its nested structure with the [flatten](https://huggingface.co/docs/datasets/process#flatten) method:

In [3]:
eli5 = eli5.flatten()

eli5["train"][0]

{'q_id': '5lyheh',
 'title': 'The saying "feed a cold, starve a fever". Just an old wise tale or is it accurate? If so what is happening physically?',
 'selftext': 'I ask because I currently have a cold and my girlfriend keeps reiterating this saying to me, but when I was younger my Grandma would also say this. I joined Reddit fairly recently and this is my favorite thread. Hoping some of you can help clarify!',
 'category': 'Biology',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dbzeahq'],
 'answers.text': ["Its an old wife's tale. A fever is a natural response from your immune system. If you starve yourself during that you also starve your immune system. It just people have a tenancy to throw up and keep nothing down during a fever so they tend to feed them less/they eat less. Not throwing up is important because it dehydrates you."],
 'answers.score': [4],
 'answers.text_urls': [[]],
 'title_urls': ['url'],
 'selftext_urls': ['url']}

Each subfield is now a separate column as indicated by the <u>answers</u> prefix, and the <u>text</u> field is a list now. Instead of tokenizing each sentence separately, convert the list to a string so you can jointly tokenize them.

Here is a first preprocessing function to join the list of strings for each example and tokenize the result:

In [4]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

To apply this preprocessing function over the entire dataset, use the 🤗 Datasets [map](https://huggingface.co/docs/datasets/v4.0.0/en/package_reference/main_classes#datasets.Dataset.map) method. You can speed up the <u>map</u> function by setting <u>batched=True</u> to process multiple elements of the dataset at once, and increasing the number of processes with <u>num_proc</u>. Remove any columns you don’t need:

In [5]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (8350 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2051 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1417 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (4271 > 1024). Running this sequence through the model will result in indexing errors
Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1647 > 1024). Running this sequence

This dataset contains the token sequences, but some of these are longer than the maximum input length for the model.

You can now use a second preprocessing function to
- concatenate all the sequences
- split the concatenated sequences into shorter chunks defined by <u>block_size</u>, which should be both shorter than the maximum input length and short enough for your GPU RAM.

In [6]:
block_size = 128

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

# Apply the group_texts function over the entire dataset:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4): 100%|██████████| 1000/1000 [00:00<00:00, 5505.87 examples/s]


Now create a batch of examples using [DataCollatorForLanguageModeling](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/data_collator#transformers.DataCollatorForLanguageModeling). It’s more efficient to dynamically pad the sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

In [7]:
# Use the end-of-sequence token as the padding token and set mlm=False.
# This will use the inputs as labels shifted to the right by one element:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#### Train

You’re ready to start training your model now! Load DistilGPT2 with [AutoModelForCausalLM](https://huggingface.co/docs/transformers/v4.53.3/en/model_doc/auto#transformers.AutoModelForCausalLM):

In [8]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

At this point, only three steps remain:

1. Define your training hyperparameters in [TrainingArguments](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.TrainingArguments). The only required parameter is output_dir which specifies where to save your model. You’ll push this model to the Hub by setting push_to_hub=True (you need to be signed in to Hugging Face to upload your model).

2. Pass the training arguments to [Trainer](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer) along with the model, datasets, and data collator.

3. Call [train()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer.train) to finetune your model.

In [9]:
training_args = TrainingArguments(
    output_dir="distilgpt2-eli5-clm",
    eval_strategy="epoch",
    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

/tmp/ipykernel_21536/2137829738.py:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.907900,3.856307
2,3.811900,3.846071
3,3.766200,3.842039
4,3.706200,3.843687
5,3.672600,3.846003
6,3.639000,3.846196
7,3.622900,3.853257
8,3.596300,3.854942
9,3.582000,3.857312
10,3.579800,3.858555


TrainOutput(global_step=13140, training_loss=3.687434675196353, metrics={'train_runtime': 1917.4809, 'train_samples_per_second': 54.796, 'train_steps_per_second': 6.853, 'total_flos': 3431806198087680.0, 'train_loss': 3.687434675196353, 'epoch': 10.0})

Once training is completed, use the [evaluate()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer.evaluate) method to evaluate your model and get its perplexity:

In [10]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 47.40


Then share your model to the Hub with the [push_to_hub()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer.push_to_hub) method so everyone can use your model:

In [11]:
trainer.push_to_hub()

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   0%|          |  548kB /  328MB, 1.37MB/s  
Processing Files (0 / 1)                :   1%|          | 2.20MB /  328MB, 3.66MB/s  
Processing Files (0 / 1)                :   1%|          | 3.29MB /  328MB, 4.11MB/s  
Processing Files (0 / 1)                :   2%|▏         | 7.69MB /  328MB, 7.69MB/s  
Processing Files (0 / 1)                :   4%|▍         | 12.6MB /  328MB, 10.5MB/s  
Processing Files (0 / 1)                :   5%|▍         | 14.8MB /  328MB, 10.6MB/s  




Processing Files (0 / 2)                :   6%|▌         | 19.2MB /  328MB, 12.0MB/s  


Processing Files (0 / 2)                :   7%|▋         | 23.1MB /  328MB, 12.8MB/s  


Processing Files (0 / 2)                :   8%|▊         | 25.9MB /  328MB, 12.9MB/s  


Processing Files (0 / 2)                :   9%|▊         | 28.6MB /  328MB, 13.0MB/s  


Processing Files (0 / 2)          

CommitInfo(commit_url='https://huggingface.co/koh43/distilgpt2-eli5-clm/commit/c4c04831d7d495e4745a6c30327f46473780a0ec', commit_message='End of training', commit_description='', oid='c4c04831d7d495e4745a6c30327f46473780a0ec', pr_url=None, repo_url=RepoUrl('https://huggingface.co/koh43/distilgpt2-eli5-clm', endpoint='https://huggingface.co', repo_type='model', repo_id='koh43/distilgpt2-eli5-clm'), pr_revision=None, pr_num=None)

#### Inference

The simplest way to try out your finetuned model for inference is to use it in a [pipeline()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/pipelines#transformers.pipeline). Instantiate a pipeline for text generation with your model, and pass your text to it:

In [12]:
# Prompt you’d like to generate text from...
prompt = "Somatic hypermutation allows the immune system to"

from transformers import pipeline

generator = pipeline("text-generation", model="koh43/distilgpt2-eli5-clm")
generator(prompt)

Device set to use cuda:0


[{'generated_text': "Somatic hypermutation allows the immune system to detect multiple different diseases. If you have a disease that involves multiple sclerosis (a rare autoimmune disease), you're probably not going to be able to diagnose it. This means that your immune system is not really fighting off an autoimmune disease and it's not really fighting off an autoimmune disease. The immune system is basically just a bunch of cells that make up the immune system. As the immune system gets smarter, the more cells they get to target. As the immune system becomes smarter, the more cells you have to attack. In contrast, an individual immune system can't fight off an autoimmune disease. With regards to how the immune system works, it's not the only thing that's doing the same thing. The immune system is a sort of mirror image of a single organism. The cells that make up the body's immune system are the ones that make up the body's immune system, but when it thinks about what a particular s

In [13]:
from transformers import AutoTokenizer

# Tokenize the text and return the input_ids as PyTorch tensors:
tokenizer = AutoTokenizer.from_pretrained("koh43/distilgpt2-eli5-clm")
inputs = tokenizer(prompt, return_tensors="pt").input_ids

Use the [generate()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/text_generation#transformers.GenerationMixin.generate) method to generate text. For more details about the different text generation strategies and parameters for controlling generation, check out the [Text generation strategies](https://huggingface.co/docs/transformers/generation_strategies) page.

In [14]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("koh43/distilgpt2-eli5-clm")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)

# Decode the generated token ids back into text:
tokenizer.batch_decode(outputs, skip_special_tokens=True)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['Somatic hypermutation allows the immune system to shut down a new infection, so it can\'t be the whole thing. The same goes for your immune system, the host, even if it\'s a virus with some sort of immunity system. The immune system can respond to certain infections by "infecting" it in the gut, but that would have been the same for different kinds of viruses - some would have been able to replicate successfully while others had to fend off that virus again and again, so the immune system is not going to stop the']